In [0]:
%py

def get_param(name, default):
    try:
        return dbutils.widgets.get(name)
    except Exception:
        return default

CATALOG = get_param("catalog", "pf")
LANDING = f"/Volumes/pf/landing/files/*/page_*.csv"
BRONZE  = f"{CATALOG}.bronze.models_raw"

from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, StringType, LongType,
                               BooleanType, TimestampType, DateType)

In [0]:
%py

CSV_SCHEMA = StructType([
    StructField("_id",            StringType(),  True),
    StructField("id",             StringType(),  True),
    StructField("modelId",        StringType(),  True),
    StructField("likes",          StringType(),  True),   # se castean luego
    StructField("private",        StringType(),  True),
    StructField("downloads",      StringType(),  True),
    StructField("pipeline_tag",   StringType(),  True),
    StructField("library_name",   StringType(),  True),
    StructField("createdAt",      StringType(),  True),
    StructField("lastModified",   StringType(),  True),
    StructField("tags",           StringType(),  True),
    StructField("payload_json",   StringType(),  True),
    StructField("ingesta_run_id", StringType(),  True),
    StructField("ingesta_mode",   StringType(),  True),
    StructField("page_no",        StringType(),  True),
    StructField("ingestion_ts",   StringType(),  True),
    StructField("ingestion_date", StringType(),  True),
])

In [0]:
%py

df = (spark.read
      .format("csv")
      .option("header", "true")
      .option("multiLine", "true")            # los JSON dentro del CSV pueden tener saltos
      .option("quote", '"')
      .option("escape", '"')
      .option("mode", "PERMISSIVE")           # no romper el batch por campos raros
      .schema(CSV_SCHEMA)
      .load(LANDING))

print(f"Columnas: {df.columns}")
print(f"Filas leidas: {df.count()}")

In [0]:
%py

df = df.select(
    F.col("_id"),
    F.col("id"),
    F.col("modelId"),
    F.col("likes"),
    F.col("private"),
    F.col("downloads"),
    F.col("pipeline_tag"),
    F.col("library_name"),
    F.col("createdAt"),
    F.col("lastModified"),
    F.col("tags"),
    F.col("payload_json"),
    F.col("ingesta_run_id"),
    F.col("ingesta_mode"),
    F.col("page_no"),
    F.col("ingestion_ts").cast("timestamp").alias("ingestion_ts"),
    F.col("ingestion_date").cast("date").alias("ingestion_date"),
    F.lit(None).cast("string").alias("_rescued_data"),
).filter(F.col("ingestion_date").isNotNull())

In [0]:
%py

(df.write
   .mode("append")
   .partitionBy("ingestion_date")
   .format("delta")
   .saveAsTable(BRONZE))

print("Bronze actualizada. Particiones:")

spark.sql(f"SELECT ingestion_date, COUNT(*) AS n FROM {BRONZE} GROUP BY ingestion_date ORDER BY ingestion_date").show()

spark.sql(f"SELECT * FROM {BRONZE} ORDER BY downloads DESC LIMIT 5").show()